# 2. Data Preprocessing Pipeline
## Vietnam Real Estate Dataset

## 2.1 Import Libraries

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 2.2 Load Data

In [ ]:
# Load the dataset
df = pd.read_csv('../vietnam_housing_dataset.csv')

print(f'Dataset shape: {df.shape}')
print(f'Records: {len(df):,}')

## 2.3 Handle Missing Values

In [ ]:
# Check missing values before processing
print('=== MISSING VALUES BEFORE PROCESSING ===\n')
missing_before = df.isnull().sum()
missing_pct_before = (missing_before / len(df) * 100).round(2)
print(pd.DataFrame({
    'Missing Count': missing_before,
    'Missing %': missing_pct_before
})[missing_before > 0].sort_values('Missing %', ascending=False))

In [ ]:
# Define missing value handling strategies
# Numerical columns: fill with median
numerical_cols = ['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms']

# Categorical columns: fill with mode
categorical_cols = ['House direction', 'Balcony direction', 'Legal status', 'Furniture state']

# Store original counts
df_original = df.copy()

# Fill numerical missing values with median
for col in numerical_cols:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'{col}: filled with median = {median_val}')

# Fill categorical missing values with mode
for col in categorical_cols:
    if col in df.columns:
        mode_val = df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown'
        df[col] = df[col].fillna(mode_val)
        print(f'{col}: filled with mode = {mode_val}')

In [ ]:
# Verify missing values after filling
print('\n=== MISSING VALUES AFTER FILLING ===\n')
missing_after = df[numerical_cols + categorical_cols].isnull().sum()
print(missing_after)

## 2.4 Remove Outliers using IQR Method

In [ ]:
# IQR-based outlier removal function
def remove_outliers_iqr(df, columns, multiplier=1.5):
    """
    Remove outliers using IQR method
    Outliers are values below Q1 - 1.5*IQR or above Q3 + 1.5*IQR
    """
    df_clean = df.copy()
    outlier_info = {}
    
    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR
        
        outliers_mask = (df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)
        outlier_count = outliers_mask.sum()
        outlier_info[col] = {
            'Q1': Q1,
            'Q3': Q3,
            'IQR': IQR,
            'Lower Bound': lower_bound,
            'Upper Bound': upper_bound,
            'Outliers': outlier_count
        }
        
        df_clean = df_clean[~outliers_mask]
    
    return df_clean, outlier_info

# Columns to check for outliers
outlier_cols = ['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms', 'Price']

print('=== OUTLIER DETECTION (IQR Method) ===\n')
df_clean, outlier_info = remove_outliers_iqr(df, outlier_cols)

for col, info in outlier_info.items():
    print(f'{col}:')
    print(f'  Q1: {info["Q1"]:.2f}, Q3: {info["Q3"]:.2f}, IQR: {info["IQR"]:.2f}')
    print(f'  Bounds: [{info["Lower Bound"]:.2f}, {info["Upper Bound"]:.2f}]')
    print(f'  Outliers removed: {info["Outliers"]:,}')
    print()

In [ ]:
# Summary of data cleaning
print('=== DATA CLEANING SUMMARY ===\n')
print(f'Original records: {len(df_original):,}')
print(f'After outlier removal: {len(df_clean):,}')
print(f'Records removed: {len(df_original) - len(df_clean):,}')
print(f'Remaining: {len(df_clean) / len(df_original) * 100:.2f}%')

In [ ]:
# Visualize before and after outlier removal
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price distribution before/after
axes[0, 0].hist(df_original['Price'], bins=50, alpha=0.7, label='Before', color='red')
axes[0, 0].hist(df_clean['Price'], bins=50, alpha=0.7, label='After', color='green')
axes[0, 0].set_xlabel('Price (Billions VND)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Price Distribution: Before vs After')
axes[0, 0].legend()

# Area distribution before/after
axes[0, 1].hist(df_original['Area'], bins=50, alpha=0.7, label='Before', color='red')
axes[0, 1].hist(df_clean['Area'], bins=50, alpha=0.7, label='After', color='green')
axes[0, 1].set_xlabel('Area (m\u00b2)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Area Distribution: Before vs After')
axes[0, 1].legend()

# Boxplot before
axes[1, 0].boxplot([df_original['Area'], df_original['Price']],
                   labels=['Area', 'Price'])
axes[1, 0].set_title('Before Outlier Removal')

# Boxplot after
axes[1, 1].boxplot([df_clean['Area'], df_clean['Price']],
                   labels=['Area', 'Price'])
axes[1, 1].set_title('After Outlier Removal')

plt.tight_layout()
plt.savefig('../figures/10_outlier_removal.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n>>> Chart saved: ../figures/10_outlier_removal.png')

## 2.5 Feature Engineering: Extract City and District

In [ ]:
# Feature engineering functions
def extract_city(address):
    if pd.isna(address):
        return 'Unknown'
    address = str(address).upper()
    if 'H\u00c0 N\u1ed8I' in address or 'HANOI' in address:
        return 'H\u00e0 N\u1ed9i'
    elif 'H\u1ed2 CH\u00cd MINH' in address or 'HCM' in address or 'TP.HCM' in address:
        return 'H\u1ed3 Ch\u00ed Minh'
    elif '\u0110\u00c0 N\u1eb8NG' in address or 'DANANG' in address:
        return '\u0110\u00e0 N\u1eb5ng'
    elif 'H\u1ea2I PH\u00d2NG' in address:
        return 'H\u1ea3i Ph\u00f2ng'
    elif 'C\u1ea6N TH\u01a0' in address:
        return 'C\u1ea7n Th\u01a1'
    elif 'H\u01afNG Y\u00caN' in address:
        return 'H\u01b0ng Y\u00ean'
    elif 'B\u00ccNH D\u01af\u01a0NG' in address:
        return 'B\u00ecnh D\u01b0\u01a1ng'
    elif '\u0110\u1ed2NG NAI' in address:
        return '\u0110\u1ed3ng Nai'
    elif 'QU\u1ea2NG NINH' in address:
        return 'Qu\u1ea3ng Ninh'
    elif 'H\u1ea2I D\u01af\u01a0NG' in address:
        return 'H\u1ea3i D\u01b0\u01a1ng'
    elif 'PH\u00da TH\u1ecc' in address:
        return 'Ph\u00fa Th\u1ecd'
    else:
        return 'Other'

def extract_district(address):
    """Extract district from address string"""
    if pd.isna(address):
        return 'Unknown'
    address = str(address)
    
    # Try to find district patterns
    import re
    
    # Pattern: Ph\u01b0\u1eddng X or Qu\u1eadn X
    patterns = [
        r'Ph\u01b0\u1eddng\s+([\w\s]+?)(?:,|$)',
        r'Qu\u1eadn\s+([\w\s]+?)(?:,|$)',
        r'Huy\u1ec7n\s+([\w\s]+?)(?:,|$)',
        r'X\u00e3\s+([\w\s]+?)(?:,|$)'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, address, re.IGNORECASE)
        if match:
            return match.group(1).strip()
    
    return 'Unknown'

# Apply feature engineering
df_clean['City'] = df_clean['Address'].apply(extract_city)
df_clean['District'] = df_clean['Address'].apply(extract_district)

print('=== FEATURE ENGINEERING ===\n')
print(f'Unique Cities: {df_clean["City"].nunique()}')
print(f'Unique Districts: {df_clean["District"].nunique()}')
print('\nCity Distribution:')
print(df_clean['City'].value_counts())

## 2.6 One Hot Encoding

In [ ]:
# Define columns for one-hot encoding
categorical_columns = ['House direction', 'Balcony direction', 'Legal status', 'Furniture state', 'City']

print('=== ONE HOT ENCODING ===\n')
print('Columns to encode:')
for col in categorical_columns:
    print(f'  {col}: {df_clean[col].nunique()} unique values')

# Apply one-hot encoding
df_encoded = pd.get_dummies(df_clean, columns=categorical_columns, drop_first=False)

print(f'\nShape before encoding: {df_clean.shape}')
print(f'Shape after encoding: {df_encoded.shape}')

In [ ]:
# Show encoded column names
encoded_cols = [col for col in df_encoded.columns if any(cat in col for cat in categorical_columns)]
print(f'\nEncoded columns ({len(encoded_cols)}):')
for col in encoded_cols:
    print(f'  - {col}')

## 2.7 Standard Scaling

In [ ]:
# Define numerical columns for scaling
numerical_columns = ['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms']

# Create scaler
scaler = StandardScaler()

# Fit and transform
df_scaled = df_encoded.copy()
df_scaled[numerical_columns] = scaler.fit_transform(df_encoded[numerical_columns])

print('=== STANDARD SCALING ===\n')
print('Scaling parameters:')
for i, col in enumerate(numerical_columns):
    print(f'  {col}: mean={scaler.mean_[i]:.4f}, std={scaler.scale_[i]:.4f}')

print('\nScaled statistics:')
print(df_scaled[numerical_columns].describe().round(4))

## 2.8 Save Preprocessing Artifacts

In [ ]:
# Save preprocessed data
df_scaled.to_csv('../data_preprocessed.csv', index=False)
df_clean.to_csv('../data_cleaned.csv', index=False)

# Save scaler
joblib.dump(scaler, '../models/scaler.pkl')

print('=== PREPROCESSING ARTIFACTS SAVED ===\n')
print('  - data_preprocessed.csv (scaled and encoded)')
print('  - data_cleaned.csv (cleaned, not scaled)')
print('  - models/scaler.pkl (StandardScaler)')

## 2.9 Complete Preprocessing Pipeline

In [ ]:
# Complete preprocessing pipeline function
def preprocess_pipeline(input_df):
    """
    Complete preprocessing pipeline for new data
    """
    df = input_df.copy()
    
    # 1. Handle missing values
    numerical_cols = ['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms']
    categorical_cols = ['House direction', 'Balcony direction', 'Legal status', 'Furniture state']
    
    for col in numerical_cols:
        df[col] = df[col].fillna(df[col].median())
    
    for col in categorical_cols:
        df[col] = df[col].fillna(df[col].mode()[0])
    
    # 2. Extract city
    df['City'] = df['Address'].apply(extract_city)
    df['District'] = df['Address'].apply(extract_district)
    
    # 3. One-hot encoding
    df_encoded = pd.get_dummies(df, columns=categorical_cols + ['City'], drop_first=False)
    
    # 4. Scaling
    scaler = joblib.load('../models/scaler.pkl')
    df_encoded[numerical_cols] = scaler.transform(df_encoded[numerical_cols])
    
    return df_encoded

print('Preprocessing pipeline function defined.')

## Preprocessing Summary

In [ ]:
print('='*60)
print('DATA PREPROCESSING - SUMMARY')
print('='*60)
print(f'\nOriginal records: {len(df_original):,}')
print(f'Final records: {len(df_clean):,}')
print(f'Records removed: {len(df_original) - len(df_clean):,}')
print(f'\nFeatures after preprocessing: {len(df_scaled.columns)}')
print(f'\nPipeline Steps:')
print('  1. Missing value imputation (median/mode)')
print('  2. Outlier removal (IQR, multiplier=1.5)')
print('  3. Feature engineering (City extraction)')
print('  4. One-hot encoding (House/Balcony direction, Legal status, Furniture, City)')
print('  5. Standard scaling (Area, Frontage, Access Road, Floors, Bedrooms, Bathrooms)')
print('='*60)